# Volatility targeting con bandas

> ¿Pueden las bandas de no negociación reducir los costos de volatility targeting sin deteriorar significativamente el control del riesgo?

La notebook contendrá cinco bloques, desarrollados progresivamente:

| Bloque | Contenido                                              |
| ------ | ------------------------------------------------------ |
| 01     | Datos reales de SPY y validación                       |
| 02     | Pronósticos EWMA $ \lambda = 0.94 $                   |
| 03     | Exposición objetivo y políticas P0, P1 y P2            |
| 04     | Backtest con ejecución en apertura $ (t+1) $ y costos  |
| 05     | Evaluación estadística y económica frente a benchmarks |

Parámetros iniciales: volatilidad objetivo del 12%, exposición máxima 1× y bandas de 0, 5 y 10 puntos porcentuales.



## Bloque 01 — Datos de SPY

Usamos la serie diaria **Gold** aprobada por el quality gate. Revisamos cuatro cosas sencillas:

1. Cobertura temporal.
2. Fechas únicas y ordenadas.
3. Precios de apertura, máximo, mínimo, cierre y cierre ajustado.
4. Dividendos y splits, para saber qué ajustes existen.

In [6]:
from pathlib import Path

import polars as pl

def find_project_root() -> Path:
    # Funciona si Jupyter abre la notebook desde volatility_target o desde su carpeta padre.
    for folder in (Path.cwd(), *Path.cwd().parents):
        for candidate in (folder, folder / 'volatility_target'):
            if (candidate / 'data' / 'gold' / 'daily_series' / 'etf' / 'SPY.parquet').exists():
                return candidate
    raise FileNotFoundError('No encontré data/gold/daily_series/etf/SPY.parquet. Ejecuta primero el pipeline de SPY.')


project_root = find_project_root()
spy_path = project_root / 'data' / 'gold' / 'daily_series' / 'etf' / 'SPY.parquet'
spy = pl.read_parquet(spy_path)

print(f"Filas: {spy.height:,}")
print(f"Periodo: {spy['session_date'].min()} a {spy['session_date'].max()}")

spy.select([
    'session_date', 'open', 'close', 'adjusted_close',
    'dividend_amount', 'split_coefficient',
]).tail(5)

Filas: 2,227
Periodo: 2018-01-02 a 2026-11-10


session_date,open,close,adjusted_close,dividend_amount,split_coefficient
date,f64,f64,f64,f64,f64
2026-11-04,762.99,766.81,766.81,0.0,1.0
2026-11-05,765.47,733.0,733.0,0.0,1.0
2026-11-06,734.82,735.29,735.29,0.0,1.0
2026-11-09,741.95,752.69,752.69,0.0,1.0
2026-11-10,747.94,749.92,749.92,0.0,1.0


In [7]:
price_columns = ['open', 'high', 'low', 'close', 'adjusted_close']

checks = {
    'Fechas únicas': spy['session_date'].n_unique() == spy.height,
    'Fechas ordenadas': spy['session_date'].to_list() == sorted(spy['session_date'].to_list()),
    'Sin precios faltantes': sum(spy[column].null_count() for column in price_columns) == 0,
    'OHLC coherente': spy.select(
        ((pl.col('high') >= pl.max_horizontal('open', 'close', 'low')) &
         (pl.col('low') <= pl.min_horizontal('open', 'close', 'high'))).all()
    ).item(),
}

for name, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {name}")

print(f"Sesiones con dividendo: {spy.filter(pl.col('dividend_amount') > 0).height}")
print(f"Sesiones con split: {spy.filter(pl.col('split_coefficient') != 1).height}")

✓ Fechas únicas
✓ Fechas ordenadas
✓ Sin precios faltantes
✓ OHLC coherente
Sesiones con dividendo: 35
Sesiones con split: 0



La serie Gold de SPY cubre del **2018-01-02** al **2026-11-10** y contiene **2,227** sesiones. La auditoría aprobó la serie: no hay fechas duplicadas, precios faltantes ni inconsistencias OHLC. Hay 35 sesiones con dividendos y ningún split en este periodo.

Por tanto, tenemos precios de apertura, cierre y cierre ajustado suficientes para continuar después con EWMA, sin implementar todavía ninguna política de rebalanceo.

## Bloque 02 — EWMA Forecasting

<b> Objetivo: </b> Construir y comprender una serie histórica de pronósticos de volatilidad de SPY, utilizando EWMA y únicamente información disponible en cada fecha.

Nuestra ecuación será:

$$  \boxed{ \widehat{\sigma}\_{t+1|t}^{2} =0.94\widehat{\sigma}\_{t|t-1}^{2} +0.06r_t^2 }  $$

### Alcance exacto

| Paso | Actividad                                                      |
| ---- | -------------------------------------------------------------- |
| 02.1 | Calcular retornos diarios simples a partir de `adjusted_close` |
| 02.2 | Inicializar EWMA con las primeras 20 observaciones válidas     |
| 02.3 | Calcular recursivamente la varianza pronosticada               |
| 02.4 | Obtener volatilidad diaria y anualizada                        |
| 02.5 | Visualizar e interpretar la evolución del pronóstico           |

Para mantener la coherencia matemática, utilizaremos media condicional cero y una varianza inicial calculada como el promedio de los primeros 20 rendimientos al cuadrado.

La primera predicción se emitirá después de esas 20 observaciones. Las sesiones anteriores no tendrán pronóstico.

### Instrucción para implementar exclusivamente el Bloque 02

Implementación acotada

Añade un nuevo bloque a la notebook existente `01_volatility_targeting.ipynb`.

Reutiliza el DataFrame Polars `spy`. No modifiques el Bloque 01.

Implementa retornos ajustados, EWMA recursivo con λ = 0.94 e inicialización de 20 retornos válidos. Cada pronóstico fechado en \\(t\\) debe referirse a la sesión siguiente.

Presenta una tabla con fecha, retorno observado, varianza pronosticada, volatilidad diaria y volatilidad anualizada. Añade una figura Plotly de volatilidad anualizada y una breve conclusión descriptiva.

Valida que las varianzas sean no negativas, que no haya pronósticos antes de disponer de 20 retornos y que la recurrencia no utilice rendimientos futuros.

No añadas: exposición objetivo, políticas P0/P1/P2, operaciones, comisiones, backtest ni análisis de rentabilidad.

Hay un detalle metodológico que conviene conservar para después: `adjusted_close` incorpora dividendos, pero nuestra ejecución utilizará aperturas sin ajustar. En el bloque de backtesting tendremos que reconciliar correctamente los flujos por dividendos; no bastará con mezclar retornos ajustados y precios sin ajustar. Eso no bloquea el cálculo de EWMA.

## ¿Qué vamos a estudiar cuando termine?

No quiero que nos limitemos a observar una gráfica atractiva. Evaluaremos tres cuestiones:

1. ¿Cómo responde EWMA cuando SPY experimenta movimientos extremos?
2. ¿Cómo se comporta el pronóstico durante periodos tranquilos?
3. ¿Qué información estamos obteniendo realmente: una estimación de riesgo o evidencia de capacidad predictiva?

Esa tercera distinción es crucial: todavía no habremos demostrado precisión fuera de muestra.

Adelante con el Bloque 02. Cuando tengas la tabla, figura y validaciones, compárteme los resultados. Los interpretaremos juntos antes de avanzar al cálculo de exposición.

In [8]:
import math

LAMBDA = 0.94
WARMUP_RETURNS = 20
TRADING_DAYS = 252

# r_t usa sólo el cierre ajustado observado en la fecha t.
returns = (
    spy.select(
        'session_date',
        pl.col('adjusted_close').pct_change().alias('adjusted_return'),
    )
    .drop_nulls()
)

return_dates = returns['session_date'].to_list()
return_values = returns['adjusted_return'].to_list()
forecast_variance = [None] * len(return_values)

# Tras observar los primeros 20 retornos, emitimos el primer pronóstico para t+1.
forecast_variance[WARMUP_RETURNS - 1] = sum(
    value ** 2 for value in return_values[:WARMUP_RETURNS]
) / WARMUP_RETURNS

# El pronóstico de cada fila se forma con información disponible hasta esa misma fecha.
for index in range(WARMUP_RETURNS, len(return_values)):
    forecast_variance[index] = (
        LAMBDA * forecast_variance[index - 1]
        + (1 - LAMBDA) * return_values[index] ** 2
    )

ewma = (
    pl.DataFrame({
        'signal_date': return_dates,
        'forecast_for_date': return_dates[1:] + [None],
        'adjusted_return': return_values,
        'forecast_variance': forecast_variance,
    })
    .with_columns(
        pl.col('forecast_variance').sqrt().alias('forecast_vol_daily'),
        (pl.col('forecast_variance').sqrt() * math.sqrt(TRADING_DAYS)).alias(
            'forecast_vol_annualized'
        ),
    )
    .filter(pl.col('forecast_variance').is_not_null())
    .filter(pl.col('forecast_for_date').is_not_null())
)

ewma.select([
    'signal_date', 'forecast_for_date', 'adjusted_return',
    'forecast_variance', 'forecast_vol_daily', 'forecast_vol_annualized',
]).head()

signal_date,forecast_for_date,adjusted_return,forecast_variance,forecast_vol_daily,forecast_vol_annualized
date,date,f64,f64,f64,f64
2018-01-31,2018-02-01,0.000497,0.000034,0.00587,0.093187
2018-02-01,2018-02-02,-0.001135,0.000032,0.005698,0.090455
2018-02-02,2018-02-05,-0.02177,0.000059,0.007678,0.12189
2018-02-05,2018-02-06,-0.041822,0.00016,0.012664,0.201028
2018-02-06,2018-02-07,0.019702,0.000174,0.013192,0.20942


In [11]:
initial_variance = sum(value ** 2 for value in return_values[:WARMUP_RETURNS]) / WARMUP_RETURNS
recurrence_is_correct = all(
    math.isclose(
        forecast_variance[index],
        LAMBDA * forecast_variance[index - 1] + (1 - LAMBDA) * return_values[index] ** 2,
        rel_tol=1e-12,
    )
    for index in range(WARMUP_RETURNS, len(return_values))
)

checks = {
    'Sin pronóstico antes de 20 retornos': all(
        value is None for value in forecast_variance[:WARMUP_RETURNS - 1]
    ),
    'Varianzas no negativas': ewma.select((pl.col('forecast_variance') >= 0).all()).item(),
    'Inicialización correcta': math.isclose(
        forecast_variance[WARMUP_RETURNS - 1], initial_variance, rel_tol=1e-12
    ),
    'Recurrencia sin rendimientos futuros': recurrence_is_correct,
}

for name, passed in checks.items():
    print(f"{'ok' if passed else 'no'} {name}")

print(f"Pronósticos históricos disponibles: {ewma.height:,}")
print(f"Primer pronóstico: {ewma['signal_date'][0]} para {ewma['forecast_for_date'][0]}")

ok Sin pronóstico antes de 20 retornos
ok Varianzas no negativas
ok Inicialización correcta
ok Recurrencia sin rendimientos futuros
Pronósticos históricos disponibles: 2,206
Primer pronóstico: 2018-01-31 para 2018-02-01


In [10]:
import plotly.graph_objects as go

figure = go.Figure(
    go.Scatter(
        x=ewma['forecast_for_date'].to_list(),
        y=ewma['forecast_vol_annualized'].to_list(),
        mode='lines',
        line={'color': '#2563eb', 'width': 1.5},
        name='EWMA',
    )
)
figure.update_layout(
    title='SPY: volatilidad anualizada pronosticada por EWMA',
    xaxis_title='Sesión para la que se emite el pronóstico',
    yaxis_title='Volatilidad anualizada',
    yaxis_tickformat='.0%',
    template='plotly_white',
    height=420,
    margin={'l': 50, 'r': 25, 't': 60, 'b': 50},
)
figure.show()

latest_forecast = ewma.tail(1).to_dicts()[0]
print(
    f"Último pronóstico disponible: {latest_forecast['forecast_vol_annualized']:.1%} "
    f"para la sesión posterior a {latest_forecast['signal_date']}."
)
print('EWMA reacciona a movimientos grandes y suaviza episodios aislados; todavía no mide precisión fuera de muestra.')

Último pronóstico disponible: 21.9% para la sesión posterior a 2026-11-09.
EWMA reacciona a movimientos grandes y suaviza episodios aislados; todavía no mide precisión fuera de muestra.
